# 01. RAG 与检索增强

RAG (Retrieval-Augmented Generation) 是 Agent 记忆系统最常用的实现模式：

1. **离线**：文档 → 切分 → 向量化 → 存入索引
2. **在线**：查询 → 向量化 → 检索 top-K → 注入 context → 生成回答

本 notebook 用纯 Python 实现 TF-IDF 检索器，理解 RAG 的核心机制。

> **检查点**：能画出 RAG 的离线索引和在线查询两条 pipeline。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# 从零实现 TF-IDF 检索器
import math
from collections import Counter

class TFIDFRetriever:
    def __init__(self):
        self.docs = []
        self.doc_tokens = []
    
    def add(self, doc_id: str, text: str):
        self.docs.append((doc_id, text))
        self.doc_tokens.append(self._tokenize(text))
    
    def _tokenize(self, text: str) -> list[str]:
        return text.lower().replace(".","").replace(",","").split()
    
    def search(self, query: str, top_k: int = 3) -> list[tuple[str,float]]:
        query_tokens = self._tokenize(query)
        N = len(self.docs)
        df = Counter()
        for tokens in self.doc_tokens:
            df.update(set(tokens))
        
        scores = []
        for i, tokens in enumerate(self.doc_tokens):
            tf = Counter(tokens)
            score = 0.0
            for tok in query_tokens:
                if tok in tf:
                    idf = math.log((N+1)/(df.get(tok,0)+1)) + 1
                    score += tf[tok] * idf
            scores.append((self.docs[i][0], score))
        scores.sort(key=lambda x: -x[1])
        return scores[:top_k]

r = TFIDFRetriever()
for i, text in enumerate([
    "Python is great for data science and machine learning",
    "Rust provides memory safety without garbage collection",
    "JavaScript runs in browsers and on servers with Node.js",
    "Python has excellent libraries like NumPy and PyTorch",
    "Go is designed for concurrency and cloud services",
]):
    r.add(f"doc_{i}", text)

results = r.search("python machine learning", top_k=3)
print("Query: 'python machine learning'")
for doc_id, score in results:
    print(f"  [{score:.3f}] {doc_id}")

Query: 'python machine learning'
  [5.890] doc_0
  [1.693] doc_3
  [0.000] doc_1


## RAG 的关键工程决策

| 决策点 | 选项 | 权衡 |
|--------|------|------|
| Chunk 大小 | 256 / 512 / 1024 tokens | 小=精确但缺上下文，大=完整但噪声多 |
| Embedding 模型 | text-embedding-3-small / bge-large / Cohere | 维度、多语言、成本 |
| Top-K | 3 / 5 / 10 | 太少=遗漏，太多=噪声+成本 |
| Re-ranking | Cohere Rerank / cross-encoder | 提升精度但增加延迟 |

> **检查点**：能说出 chunk size 为 256 和 1024 时各自的优缺点。

## 真实 API 实验

用上面实现的 TF-IDF 检索器取回文档，让真实 LLM 分别在有/无检索内容时回答，对比 grounding 差异。


In [3]:
# RAG 对比实验（复用本 notebook 前面的 TFIDFRetriever）
docs = [
    "ReAct 是 2022 年 Yao 等人提出的 agent 范式，交替进行推理与行动。",
    "Tree of Thoughts 把推理组织成搜索树，支持回溯与剪枝。",
    "Reflexion 通过文字反思把失败经验存入情节记忆。",
]
retriever = TFIDFRetriever()  # 前面 cell 已定义的类
for i, d in enumerate(docs):
    retriever.add(f"rag_doc_{i}", d)  # 前面 cell 的接口为 add(doc_id, text)

q = "ReAct 是谁提出的？核心机制是什么？"
hits = retriever.search(q, top_k=1)  # 前面 cell 的接口为 search(query, top_k)，返回 [(doc_id, score)]
print("检索命中:", hits)

no_rag = chat(client, q, max_tokens=150)
with_rag = chat(client, f"参考文档：{docs[0]}\n\n回答：{q}", max_tokens=150)
print("\n【无 RAG】", no_rag)
print("\n【有 RAG】", with_rag)


检索命中: [('rag_doc_0', 1.6931471805599454)]



【无 RAG】 ReAct 是由 **Shunyu Yao** 等人在 2022 年的论文《ReAct: Synergizing Reasoning and Acting in Language Models》中提出的。  

其核心机制是 **推理

【有 RAG】 ReAct 是 2022 年由 Yao 等人提出的，其核心机制是交替进行推理与行动。


## 练习

1. 给 TFIDFRetriever 增加 metadata filtering：只搜索指定 tag 的文档。
2. 实现 BM25 评分（TF-IDF 的改进版，处理文档长度归一化），对比检索结果。
3. 思考：如果检索到的文档恰好包含 prompt injection 内容（如「忽略之前指令」），Agent 会怎样？如何防御？

> **检查点**：能用 TF-IDF 或 BM25 为一个 100 篇文档的知识库实现可用的检索功能。